# Fourier Coefficients of Periodic Splines
We display in thick gray the realization of a random periodic polynomial spline of a specified period, degree, and delay; the red stemlines indicate the extent of one period. We then plot in thinner blue its approximation by a truncated list of its Fourier-series coefficients.

In [ ]:
# Load the required libraries
import cmath
import ipywidgets as widgets
import math
import matplotlib.pyplot as plt
import numpy as np
import warnings

import splinekit as sk # This library

# Setup
max_period = 15 # Maximal period
max_degree = 9 # Maximal spline degree
max_delay = 3.0 # Maximal absolute delay
max_coeffs = 25 # Maximal number of Fourier coefficients

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Initial random periodic cubic spline
s0 = sk.PeriodicSpline1D.from_spline_coeff(rng.standard_normal(6), degree = 3)

# Plot
def update_plot (
    period = 6,
    degree = 3,
    delay = 0.0,
    coeffs = 5
):
    global s0

    # Update of the spline
    if s0.period != period:
        s0 = sk.PeriodicSpline1D.from_spline_coeff(
            rng.standard_normal(period),
            degree = s0.degree
        )
    s0.degree = degree
    s0.delay = delay

    # Fourier coefficients
    nu0 = (coeffs - 1) // 2
    c = np.array(
        [s0.fourier_coeff(nu) for nu in range(-nu0, nu0 + 1)],
        dtype = complex
    )

    # Domain
    plotdomain = sk.interval.Closed((-0.5, period + 0.5))

    # Fourier approximation data
    # Location of the samples
    abscissa = np.linspace(
        plotdomain.infimum,
        plotdomain.supremum,
        num = 200 + 1,
        dtype = float
    )
    with warnings.catch_warnings(action = "ignore"): # To silence '.real'
        f_data = np.array(
            [
                math.fsum([
                    cnu * cmath.exp(1j * (nu - nu0) * (2.0 * math.pi / period) * x)
                    for (nu, cnu) in enumerate(c)
                ]).real
                for x in abscissa
            ],
            dtype = float
        )

    # Plot of the spline
    subplot = plt.subplots()
    s0.plot(
        subplot,
        plotdomain = plotdomain,
        plotpoints = 200 + 1,
        curve_fmt = "#E0E0E0",
        curve_lw = 7.0,
        curve_markerfmt = "oC7",
        curvestem_linefmt = "-C7",
        knot_marker = " "
    )

    # Plot of the Fourier approximation
    plt.plot(abscissa, f_data, "-C0", lw = 0.75)

    # Final display
    plt.show()

# Slider for the number of Fourier coefficients
coeffs_intslider_widget = widgets.IntSlider(
    value = 5,
    min = 1,
    max = max_coeffs,
    step = 2,
    description = "# coeffs"
)

widgets.interactive(
    update_plot,
    period = (1, max_period),
    degree = (0, max_degree),
    delay = (-max_delay, max_delay),
    coeffs = coeffs_intslider_widget
)
